# بناء تطبيق أندرويد (APK) لـ Smart View Receiver
هذا الدفتر (Notebook) مصمم ليقوم بتجهيز البيئة وبناء ملف APK للتطبيق بشكل تلقائي داخل Google Colab.

### الخطوات المطلوبة منك:
1. قم برفع ملف المشروع (الذي قمت بتحميله كملف ZIP من AI Studio) إلى لوحة الملفات في القائمة الجانبية هنا في Colab.
2. قم بإعادة تسمية الملف المرفوع ليصبح اسمه بالضبط `project.zip`.
3. قم بتشغيل الخلايا التالية بالترتيب (اضغط على زر Play بجوار كل خلية).

In [ ]:
%%bash
echo "==> 1. تحديث النظام وتثبيت Java 17 و Node.js 20..."
curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
sudo apt-get install -y nodejs openjdk-17-jdk unzip > /dev/null
echo "==> اكتمل التثبيت."

In [ ]:
%%bash
echo "==> 2. إعداد Android SDK و Tools..."
mkdir -p /content/android-sdk/cmdline-tools
cd /content
wget -q https://dl.google.com/android/repository/commandlinetools-linux-10406996_latest.zip -O cmdtools.zip
unzip -q cmdtools.zip -d /content/android-sdk/cmdline-tools
mv /content/android-sdk/cmdline-tools/cmdline-tools /content/android-sdk/cmdline-tools/latest
export ANDROID_HOME=/content/android-sdk
export PATH=$PATH:$ANDROID_HOME/cmdline-tools/latest/bin
yes | sdkmanager --licenses > /dev/null
sdkmanager "platform-tools" "platforms;android-34" "build-tools;34.0.0" > /dev/null
echo "==> تم إعداد Android SDK."

In [ ]:
%%bash
echo "==> 3. فك ضغط ملف المشروع (project.zip)..."
if [ ! -f "/content/project.zip" ]; then
  echo "[خطأ] ملف project.zip غير موجود! الرجاء رفع ملف المشروع من الشريط الجانبي وإعادة تسميته إلى project.zip"
  exit 1
fi
rm -rf /content/app_project
unzip -q /content/project.zip -d /content/app_project
echo "==> تم فك الضغط بنجاح."

In [ ]:
%%bash
echo "==> 4. تثبيت الحزم وبناء المشروع وتحويله إلى تطبيق Android عبر Capacitor..."
export ANDROID_HOME=/content/android-sdk
cd /content/app_project

echo "-> تثبيت الاعتماديات (npm install)..."
npm install
echo "-> بناء المشروع (npm run build)..."
npm run build

echo "-> تهيئة Capacitor..."
npm install @capacitor/core @capacitor/android
npm install -D @capacitor/cli

npx cap init "Smart View" "com.smartview.receiver" --web-dir dist
npx cap add android
npx cap sync android
echo "==> تم تجهيز تطبيق الأندرويد."

In [ ]:
%%bash
echo "==> 5. تجميع ملف الـ APK النهائي (Gradle Build)..."
export ANDROID_HOME=/content/android-sdk
export PATH=$PATH:$ANDROID_HOME/cmdline-tools/latest/bin
cd /content/app_project/android
chmod +x gradlew
./gradlew assembleDebug

cp app/build/outputs/apk/debug/app-debug.apk /content/SmartViewReceiver.apk
echo "--------------------------------------------------"
echo "✅ تم إنشاء ملف الـ APK بنجاح!"
echo "يمكنك الآن العثور على ملف SmartViewReceiver.apk في لوحة الملفات الجانبية. (انقر بزر الماوس الأيمن عليه ثم اختر Download)."
echo "--------------------------------------------------"